<a href="https://colab.research.google.com/github/RanemSaud/LMSGuide/blob/main/ModelGuide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip -q install sentence-transformers faiss-cpu

In [5]:
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

knowledge_base = [
    {
        "question": "كيف أرفع الواجب في Moodle؟",
        "topic": "assignment",
        "role": "student",
        "language": "ar",
        "answer": "افتح المقرر، ثم الواجب، واضغط «إضافة تسليم». ارفع الملف ثم احفظ التغييرات."
    },
    {
        "question": "How do I submit an assignment in Moodle?",
        "topic": "assignment",
        "role": "student",
        "language": "en",
        "answer": "Open the course and assignment, select Add submission, upload your file, then save."
    },
    {
        "question": "كيف أنشئ واجبًا للطلاب؟",
        "topic": "assignment",
        "role": "instructor",
        "language": "ar",
        "answer": "افتح المقرر، فعّل وضع التحرير، ثم أضف نشاط «واجب» واضبط موعد التسليم."
    },
    {
        "question": "How do I create an assignment for students?",
        "topic": "assignment",
        "role": "instructor",
        "language": "en",
        "answer": "Turn editing on in the course, add an Assignment activity, and set its deadline."
    },
    {
        "question": "كيف أبدأ الاختبار في Moodle؟",
        "topic": "quiz_exam",
        "role": "student",
        "language": "ar",
        "answer": "افتح المقرر والاختبار، تحقق من موعده، ثم اضغط «ابدأ المحاولة»."
    },
    {
        "question": "How do I start a Moodle quiz?",
        "topic": "quiz_exam",
        "role": "student",
        "language": "en",
        "answer": "Open the course and quiz, check its schedule, then select Start attempt."
    },
    {
        "question": "كيف أضيف اختبارًا جديدًا للطلاب؟",
        "topic": "quiz_exam",
        "role": "instructor",
        "language": "ar",
        "answer": "افتح المقرر، فعّل وضع التحرير، ثم أضف نشاط «اختبار» وحدد الوقت والأسئلة."
    },
    {
        "question": "How do I create a quiz for students?",
        "topic": "quiz_exam",
        "role": "instructor",
        "language": "en",
        "answer": "Turn editing on, add a Quiz activity, and configure its timing and questions."
    },
    {
        "question": "لا أستطيع تسجيل الدخول إلى Moodle",
        "topic": "login_access",
        "role": "student",
        "language": "ar",
        "answer": "تحقق من بيانات الدخول وأعد تعيين كلمة المرور. إذا استمرت المشكلة، تواصل مع الدعم التقني."
    },
    {
        "question": "I cannot log in to Moodle",
        "topic": "login_access",
        "role": "student",
        "language": "en",
        "answer": "Check your login details and reset your password. Contact technical support if the issue continues."
    },
    {
        "question": "أين أجد درجاتي في Moodle؟",
        "topic": "grades",
        "role": "student",
        "language": "ar",
        "answer": "افتح المقرر ثم اختر «الدرجات» لعرض النتائج التي نشرها مدرس المقرر."
    },
    {
        "question": "Where can I see my grades in Moodle?",
        "topic": "grades",
        "role": "student",
        "language": "en",
        "answer": "Open the course and select Grades to see the results published by your instructor."
    }
]


def mask_phone_numbers(text):
    return re.sub(r"(?<!\d)05\d{8}(?!\d)", "<PHONE>", text)


print("Loading multilingual embedding model...")

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

questions = [item["question"] for item in knowledge_base]

embeddings = model.encode(
    questions,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("MoodleGuide is ready.")


def ask_moodleguide(question, language="ar", user_role="student"):
    question = question.strip()
    language = language.strip().lower()
    user_role = user_role.strip().lower()

    if not question:
        raise ValueError("اكتب سؤالًا أولًا.")

    if language not in ("ar", "en"):
        raise ValueError("اللغة يجب أن تكون ar أو en.")

    if user_role not in ("student", "instructor"):
        raise ValueError("الدور يجب أن يكون student أو instructor.")

    safe_question = mask_phone_numbers(question)

    query_embedding = model.encode(
        [safe_question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = index.search(query_embedding, index.ntotal)

    for score, idx in zip(scores[0], indices[0]):
        item = knowledge_base[int(idx)]

        if item["language"] == language and item["role"] == user_role:
            return {
                "question": safe_question,
                "topic": item["topic"],
                "role": user_role,
                "similar_question": item["question"],
                "score": round(float(score), 4),
                "answer": item["answer"]
            }

Loading multilingual embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MoodleGuide is ready.


In [6]:
print("\n--- ModelGuide Testing ---")

queries = [
    ("كيف أرفع الواجب في Moodle؟", "ar", "student"),
    ("كيف أضيف اختبارًا جديدًا للطلاب؟", "ar", "instructor"),
    ("Where can I see my grades in Moodle?", "en", "student")
]

for question, language, role in queries:
    result = ask_moodleguide(question, language, role)

    print(f"User asks: {question}")
    print(f"Bot answers: {result['answer']}")
    print("-" * 40)


--- MoodleGuide Testing ---
User asks: كيف أرفع الواجب في Moodle؟
Bot answers: افتح المقرر، ثم الواجب، واضغط «إضافة تسليم». ارفع الملف ثم احفظ التغييرات.
----------------------------------------
User asks: كيف أضيف اختبارًا جديدًا للطلاب؟
Bot answers: افتح المقرر، فعّل وضع التحرير، ثم أضف نشاط «اختبار» وحدد الوقت والأسئلة.
----------------------------------------
User asks: Where can I see my grades in Moodle?
Bot answers: Open the course and select Grades to see the results published by your instructor.
----------------------------------------
